### Extraction of Data

In [3]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

c:\Users\Ashwin\anaconda3\envs\medibot_v2\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
#Extract text from PDF files
def load_pdf_files(data):
    loader=DirectoryLoader(data,
                           glob="*.pdf",
                           loader_cls=PyPDFLoader
    )
    documents=loader.load()
    return documents 

In [ ]:
extracted_data=load_pdf_files("Data")

In [ ]:
extracted_data

In [ ]:
len(extracted_data)

In [ ]:
from typing import List
from langchain.schema import Document

# Having what only required in data
def filer_to_minimal_docs(docs: List[Document]) -> List[Document]:
    "Given a list of document objects, return a new list of document objects containing only 'source' and 'page_content'."

    minimal_docs:List[Document]=[]
    for doc in docs:
        scr=doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={"source":scr}
            )
        )
    return minimal_docs

In [ ]:
minimal_docs=filer_to_minimal_docs(extracted_data)

In [ ]:
minimal_docs

### Chunking

In [ ]:
#Splitting the documents into smaller chunks
def text_split(minimal_docs):
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20
    )
    texts_chuck=text_splitter.split_documents(minimal_docs)
    return texts_chuck

In [ ]:
text_chunk=text_split(minimal_docs)
print(f"Number of Chunks: {len(text_chunk)}")

In [ ]:
text_chunk

### Embedding

In [6]:
from langchain_community.embeddings import HuggingFaceEmbeddings

def download_embeddings():
    "Download and retun the HuggingFace Embeddings model."

    model_name="BAAI/bge-small-en-v1.5"
    embedding=HuggingFaceEmbeddings(
        model_name=model_name,
        )
    
    return embedding

embedding=download_embeddings()

C:\Users\Ashwin\AppData\Local\Temp\ipykernel_12224\2864603172.py:7: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding=HuggingFaceEmbeddings(


In [7]:
embedding

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 512, 'do_lower_case': True, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': True, 'pooling_mode_mean_tokens': False, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='BAAI/bge-small-en-v1.5', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [ ]:
vectors=embedding.embed_query("Hello world") #checking the embedding for a sample text
vectors

In [ ]:
print("vector Length: ",len(vectors)) # checking the length of the embedding vector

### Storing Vector in pinecone(Knowledge base)

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()

In [ ]:
PINECONE_API_KEY=os.getenv("PINECONE_API_KEY")
MISTRAL_API_KEY=os.getenv("MISTRAL_API_KEY")

os.environ["PINECONE_API_KEY"]=PINECONE_API_KEY
os.environ["MISTRAL_API_KEY"]=MISTRAL_API_KEY

In [ ]:
from pinecone import Pinecone
pinecone_api_key=PINECONE_API_KEY

pc=Pinecone(api_key=pinecone_api_key)

In [ ]:
pc #Checking API

In [ ]:
from pinecone import ServerlessSpec

index_name="medical-chatbot"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws",region="us-east-1")
        )
    
index=pc.Index(index_name)

In [ ]:
from langchain_pinecone import PineconeVectorStore
# Storing the document vectors in pinecone
docsearch=PineconeVectorStore.from_documents(
    documents=text_chunk,
    embedding=embedding,
    index_name=index_name
)

### Building a retreiver

In [8]:
from langchain_pinecone import PineconeVectorStore

index_name="medical-chatbot"

docsearch=PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embedding
)

In [9]:
retriever=docsearch.as_retriever(search_type="similarity",search_kwargs={"k":3})

In [10]:
retrieved_docs=retriever.invoke("what is hypertension?")
retrieved_docs

[Document(id='5d3c380b-52d2-416a-9220-cba099a2bfc7', metadata={'source': 'Data\\Data.pdf'}, page_content='1902 GALE ENCYCLOPEDIA OF MEDICINE\nHypertension'),
 Document(id='a4f60a61-4cf5-460a-982f-977c30dde2bd', metadata={'source': 'Data\\Data.pdf'}, page_content='Definition\nRenovascular hypertension is a secondary form of\nhigh blood pressure caused by a narrowing of the renal\nartery.\nDescription\nPrimary hypertension, or high blood pressure,\naffects millions of Americans. It accounts for over\n90% of all cases of hypertension and develops without\napparent causes. It is helpful for the clinician to know\nif a secondary disease is present and may be contribut-\ning to the high pressure. If clinical tests indicate this is'),
 Document(id='039999df-b7f4-4bb1-a2b2-e466a705d51f', metadata={'source': 'Data\\Data.pdf'}, page_content='blood pressure go up. When thestress goes away,\nblood pressure usually returns to normal. These tem-\nporary increases in blood pressure are not considered

### Integration of Context to LLMs(Mistral AI)

In [11]:
import os
from mistralai import Mistral
from deep_translator import GoogleTranslator
from langdetect import detect, DetectorFactory

# --- Small-talk detection + updated multilingual pipeline ---

def is_small_talk(text: str) -> bool:
    """
    Very simple check for greetings / thanks like:
    'hi', 'hello', 'how are you', 'thanks', etc.
    """
    if not text:
        return False

    t = text.lower().strip()
    # remove basic punctuation
    for ch in [".", "!", "?", ","]:
        t = t.replace(ch, "")

    greeting_keywords = [
        "hi",
        "hello",
        "hey",
        "good morning",
        "good afternoon",
        "good evening",
        "good night",
        "how are you",
        "how r you",
        "how are u",
        "how r u",
        "how's it going",
        "hows it going",
        "what's up",
        "whats up",
        "how are things",
        "how do you do",
    ]

    thanks_keywords = [
        "thank you",
        "thanks",
        "thank u",
        "ok thanks",
        "okay thanks",
        "thanks a lot",
    ]

    for kw in greeting_keywords:
        if t == kw or kw in t:
            return True

    for kw in thanks_keywords:
        if t == kw or t.startswith(kw):
            return True

    return False

DetectorFactory.seed = 0  # ensures consistent language detection

# Helpers
_LANG_NAMES = {
    "en":"English","es":"Spanish","hi":"Hindi","ta":"Tamil","te":"Telugu","ml":"Malayalam","kn":"Kannada",
    "mr":"Marathi","bn":"Bengali","gu":"Gujarati","pa":"Punjabi","ur":"Urdu","fa":"Persian","ar":"Arabic",
    "fr":"French","de":"German","it":"Italian","pt":"Portuguese","ru":"Russian","ja":"Japanese",
    "ko":"Korean","zh-cn":"Chinese (Simplified)","zh-tw":"Chinese (Traditional)","id":"Indonesian",
    "he":"Hebrew","tr":"Turkish","vi":"Vietnamese","th":"Thai"
}

def language_name(code: str) -> str:
    return _LANG_NAMES.get(code.lower(), code)

def detect_lang(text: str) -> str:
    try:
        return detect(text) or "en"
    except Exception:
        return "en"

def _translate_safe(text: str, target: str, source: str = "auto") -> str:
    """Translation with graceful fallback (returns original on failure)."""
    if not text or target.lower() == source.lower():
        return text
    try:
        return GoogleTranslator(source=source, target=target).translate(text)
    except Exception:
        return text  # fallback

def translate(text: str, target: str, source: str = "auto") -> str:
    return _translate_safe(text, target=target, source=source)

def normalize_lang(code: str) -> str:
    mapping = {"zh-cn": "zh-CN", "zh-tw": "zh-TW", "iw": "he", "in": "id"}
    return mapping.get(code.lower(), code.lower())

def format_ctx(docs):
    parts = []
    for d in docs:
        src = d.metadata.get("source", "unknown")
        page = d.metadata.get("page", "n/a")
        parts.append(f"{d.page_content}\n[source: {src}, page {page}]")
    return "\n\n".join(parts)


# multilingual Mistral wrapper (outputs EN + original lang + input EN)
def ask_mistral_multilingual(user_text: str, retriever):
    """
    Pipeline:
      1) Detect input language
      2) Translate input -> English (if needed)  (input_english)
      3) If it's small-talk: answer locally (no LLM call)
      4) Otherwise, retrieve medical context & ask Mistral
      5) Translate answer -> input language
    Returns a dict with codes, names, and all strings.
    """
    user_text = (user_text or "").strip()
    if not user_text:
        return {
            "detected_language_code": "en",
            "detected_language_name": "English",
            "original_input": "",
            "input_english": "",
            "english_answer": "No question provided.",
            "localized_answer": "No question provided.",
        }

    # Language detection
    src_code = normalize_lang(detect_lang(user_text))
    src_name = language_name(src_code)
    needs_translation = (src_code.lower() != "en")

    # Input -> English (also returned for display)
    input_english = translate(user_text, target="en", source=src_code) if needs_translation else user_text

    # --- 1) Handle small-talk directly (no retrieval, no Mistral) ---
    if is_small_talk(input_english):
        english_answer = (
            "I'm just a medical chatbot, but I'm functioning well 😊. "
            "How can I help you with your health-related questions today?"
        )
        localized_answer = (
            translate(english_answer, target=src_code, source="en") if needs_translation else english_answer
        )
        return {
            "detected_language_code": src_code,
            "detected_language_name": src_name,
            "original_input": user_text,
            "input_english": input_english,
            "english_answer": english_answer,
            "localized_answer": localized_answer,
        }

    # --- 2) For real questions: retrieve medical context ---
    try:
        context = format_ctx(retriever.invoke(input_english))
    except Exception:
        context = ""

    # --- 3) Prompt that REFUSES non-medical questions ---
    prompt = (
        "You are a careful medical assistant. Answer ONLY from the provided medical context.\n"
        "If unsure, say you don't know. Keep the answer concise and patient-friendly.\n"
        "If the user's question is clearly not about health or medicine (for example, questions about sports, "
        "celebrities, geography, history, coding, etc.), explain briefly that you are a medical chatbot and cannot "
        "answer non-medical questions, and ask them to ask a health-related question instead.\n"
        "Give medical disclaimer at last stating that 'This is not a substitute for professional diagnosis'.\n\n"
        f"Question: {input_english}\n\nContext:\n{context}\n\nAnswer in English:"
    )

    # --- 4) Call Mistral ---
    with Mistral(api_key=os.environ.get("MISTRAL_API_KEY", "")) as client:
        res = client.chat.complete(
            model="mistral-small-latest",
            messages=[{"role": "user", "content": prompt}],
            stream=False,
        )
    english_answer = (res.choices[0].message.content or "").strip()

    # --- 5) Translate answer back to original language ---
    localized_answer = (
        translate(english_answer, target=src_code, source="en") if needs_translation else english_answer
    )

    return {
        "detected_language_code": src_code,
        "detected_language_name": src_name,
        "original_input": user_text,
        "input_english": input_english,
        "english_answer": english_answer,
        "localized_answer": localized_answer,
    }

# -------- Optional: pretty printer for scripts/notebooks (no Streamlit) --------
def print_pretty(result: dict):
    sep = "-" * 72
    print(sep)
    print(f"Detected language: {result['detected_language_name']} ({result['detected_language_code']})")
    print(sep)
    print("Original input:")
    print(result["original_input"])
    print(sep)
    print("English translation of input:")
    print(result["input_english"])
    print(sep)
    print("English answer (from Mistral):")
    print(result["english_answer"])
    print(sep)
    print(f"Answer translated back ({result['detected_language_name']}):")
    print(result["localized_answer"])
    print(sep)

    

In [15]:
# Small-talk
res1 = ask_mistral_multilingual("How are you?", retriever)
print("SMALL-TALK:", res1["localized_answer"])
print()

# Non-medical question
res2 = ask_mistral_multilingual("விராட் கோலி யார்?", retriever)
print("NON-MEDICAL:", res2["localized_answer"])
print()

# Medical question
res3 = ask_mistral_multilingual("What are the common symptoms of diabetes?", retriever)
print("MEDICAL:", res3["localized_answer"])


SMALL-TALK: I'm just a medical chatbot, but I'm functioning well 😊. How can I help you with your health-related questions today?

NON-MEDICAL: நான் ஒரு மருத்துவ சாட்போட் மற்றும் மருத்துவம் அல்லாத கேள்விகளுக்கு பதிலளிக்க முடியாது. அதற்குப் பதிலாக உடல்நலம் தொடர்பான கேள்வியைக் கேளுங்கள்.

*இது தொழில்முறை நோயறிதலுக்கு மாற்றாக இல்லை.*

MEDICAL: Common symptoms of diabetes include:
- **Excessive thirst**
- **Frequent urination**
- **Unexplained weight loss**
- **Extreme fatigue or lethargy**
- **Blurred vision**
- **Slow-healing wounds**
- **Increased hunger**
- **Recurrent infections (e.g., urinary tract infections, gum disease)**

Some people may have no symptoms, especially in early stages.

*This is not a substitute for professional diagnosis.*


In [16]:
res = ask_mistral_multilingual("Tengo fiebre y tos", retriever)
print_pretty(res)

------------------------------------------------------------------------
Detected language: Spanish (es)
------------------------------------------------------------------------
Original input:
Tengo fiebre y tos
------------------------------------------------------------------------
English translation of input:
I have fever and cough
------------------------------------------------------------------------
English answer (from Mistral):
Based on the provided context, your symptoms of fever and cough could be related to several conditions, including Q fever, bronchitis, or other respiratory infections. Q fever, for example, can cause fever, cough, and other flu-like symptoms. However, without more details, it's hard to pinpoint the exact cause.

**Please consult a healthcare professional for proper evaluation and treatment.**

*This is not a substitute for professional diagnosis.*
------------------------------------------------------------------------
Answer translated back (Spanish)

In [17]:
res = ask_mistral_multilingual("வாந்தியை எப்படி நிறுத்துவது", retriever)
print_pretty(res)

------------------------------------------------------------------------
Detected language: Tamil (ta)
------------------------------------------------------------------------
Original input:
வாந்தியை எப்படி நிறுத்துவது
------------------------------------------------------------------------
English translation of input:
How to stop vomiting
------------------------------------------------------------------------
English answer (from Mistral):
To stop vomiting, try these steps:
1. **Get fresh air** or move away from triggers.
2. **Eat small amounts** of olives, crackers, or suck on a lemon to calm your stomach.
3. **Sip clear fluids** like weak tea, diluted juice, or sports drinks to prevent dehydration.
4. **Eat small, frequent meals** and avoid heavy meals.
5. **Stay upright** for 45 minutes after eating.
6. **Avoid iron supplements** if they cause nausea.

If vomiting persists, seek medical help for anti-nausea meds or IV fluids.

*This is not a substitute for professional diagnosis